# 04 — BM25 Retrieval

This notebook implements the project's BM25 retrieval baseline using a fixed top-k. It indexes unique passages and preserves passage identifiers and article metadata. Retrieval features are generated separately for train, validation and test.

## 1. BM25 retrieval concept

BM25 is a sparse lexical ranking method. It scores documents using query-term matches, term frequency, inverse document frequency and document-length normalization. We use a fixed top-k for all strategies so the comparison remains controlled.

We index unique passages, retaining `paragraph_id`, article title and source metadata.

In [ ]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi

SEED=42
ROOT=Path.cwd()
if ROOT.name=="notebooks": ROOT=ROOT.parent
SPLITS=ROOT/"data"/"splits"
PROCESSED=ROOT/"data"/"processed"
RESULTS=ROOT/"results"
RESULTS.mkdir(exist_ok=True)

def load_split(name):
    df=pd.read_csv(SPLITS/f"{name}.csv")
    df["answers"]=df["answers"].apply(json.loads)
    df["is_answerable"]=df["is_answerable"].astype(bool)
    return df

train_df, val_df, test_df = [load_split(x) for x in ["train","validation","test"]]
print(len(train_df),len(val_df),len(test_df))

## 2. Tokenize text

BM25 needs tokenized terms. This simple tokenizer lowercases text and extracts alphanumeric tokens. It is intentionally transparent; later, you can compare it with a tokenizer that handles punctuation, stop words or stemming.

In [ ]:
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

def tokenize(text):
    return TOKEN_PATTERN.findall(str(text).lower())

example = train_df.iloc[0]
print("Question:", example.question)
print("Tokens:", tokenize(example.question)[:30])

## 3. Build unique passage corpora

Each split gets its own corpus, built from the passages associated with questions in that split. This prevents article-group information from crossing partitions. It also means each query is evaluated against the corresponding split's document collection; report this protocol in the final report.

In [ ]:
def make_passage_corpus(df):
    corpus = (df[["paragraph_id","article_title","context"]]
              .drop_duplicates("paragraph_id")
              .reset_index(drop=True))
    corpus["tokens"] = corpus.context.map(tokenize)
    return corpus

corpora = {
    "train": make_passage_corpus(train_df),
    "validation": make_passage_corpus(val_df),
    "test": make_passage_corpus(test_df),
}
for name, corpus in corpora.items():
    print(name, "passages:",len(corpus))

## 4. Index with BM25 and retrieve top-k

We use `rank_bm25`'s `BM25Okapi`. `get_scores` returns a score for every passage; `argsort` identifies the highest-ranked passages. BM25 scores are corpus- and query-dependent and should not be interpreted as probabilities.

In [ ]:
TOP_K = 5

class BM25Retriever:
    def __init__(self, corpus):
        self.corpus = corpus.reset_index(drop=True)
        self.tokenized = self.corpus["tokens"].tolist()
        self.index = BM25Okapi(self.tokenized)

    def retrieve(self, question, top_k=TOP_K):
        q_tokens = tokenize(question)
        scores = self.index.get_scores(q_tokens)
        order = np.argsort(scores)[::-1][:top_k]
        results = self.corpus.iloc[order][
            ["paragraph_id","article_title","context"]
        ].copy()
        results["bm25_score"] = scores[order]
        results["rank"] = np.arange(1, len(results)+1)
        return results.reset_index(drop=True)

retrievers = {name:BM25Retriever(corpus) for name,corpus in corpora.items()}

sample_question = train_df.iloc[0].question
display(retrievers["train"].retrieve(sample_question, top_k=TOP_K)[
    ["rank","bm25_score","article_title","paragraph_id","context"]
])

## 5. Batch retrieval features

For every question, calculate the top score, second score, score gap, mean top-k score, and whether its original paired passage appears in top-k. The last feature is a retrieval diagnostic, not a model confidence.

In [ ]:
def retrieve_features(df, retriever, top_k=TOP_K):
    rows=[]
    for row in df.itertuples(index=False):
        hits = retriever.retrieve(row.question, top_k=top_k)
        scores = hits.bm25_score.to_numpy()
        top_score = float(scores[0]) if len(scores) else 0.0
        second_score = float(scores[1]) if len(scores)>1 else 0.0
        rows.append({
            "question_id": row.question_id,
            "top_bm25_score": top_score,
            "second_bm25_score": second_score,
            "bm25_score_gap": top_score-second_score,
            "mean_topk_bm25": float(scores.mean()) if len(scores) else 0.0,
            "source_passage_retrieved": row.paragraph_id in set(hits.paragraph_id),
            "retrieved_passage_ids": hits.paragraph_id.tolist(),
        })
    return pd.DataFrame(rows)

train_retrieval = retrieve_features(train_df, retrievers["train"])
val_retrieval = retrieve_features(val_df, retrievers["validation"])
test_retrieval = retrieve_features(test_df, retrievers["test"])

for name, feats in [("train",train_retrieval),("validation",val_retrieval),("test",test_retrieval)]:
    print(name, "source passage in top-k:", f"{feats.source_passage_retrieved.mean():.3f}")

## 6. Inspect retrieval quality

Inspect a few questions and their top passages. Pay attention to cases where the paired source passage is not retrieved. BM25's raw score is not calibrated across different queries, which is why we tune a threshold on validation data rather than choosing an arbitrary number.

In [ ]:
def inspect_query(df, retriever, index=0, top_k=TOP_K):
    row=df.iloc[index]
    print("QUESTION:",row.question)
    print("SOURCE ARTICLE:",row.article_title)
    print("SOURCE PARAGRAPH:",row.paragraph_id)
    print("SQuAD answerable:",row.is_answerable)
    hits=retriever.retrieve(row.question, top_k)
    display(hits[["rank","bm25_score","article_title","paragraph_id"]])
    for h in hits.itertuples():
        print("\n[RANK",h.rank,"| SCORE",round(h.bm25_score,3),"]",h.paragraph_id)
        print(h.context[:500])

inspect_query(train_df, retrievers["train"], index=0)

## 7. Save retrieval outputs

Save per-question retrieval features for each split. The complete ranked passage lists are also saved as JSONL for later QA inference and error inspection.

In [ ]:
def save_retrieval(df, feats, name):
    merged=df.merge(feats, on="question_id", validate="one_to_one")
    merged.to_csv(RESULTS/f"{name}_retrieval_features.csv", index=False)
    with (RESULTS/f"{name}_retrieval_features.jsonl").open("w",encoding="utf-8") as f:
        for row in merged.to_dict(orient="records"):
            f.write(json.dumps(row,ensure_ascii=False,default=str)+"\n")
    return merged

train_retrieved=save_retrieval(train_df,train_retrieval,"train")
val_retrieved=save_retrieval(val_df,val_retrieval,"validation")
test_retrieved=save_retrieval(test_df,test_retrieval,"test")
print("Saved retrieval feature files to",RESULTS)

## 8. Questions

1. Why are BM25 scores not probabilities?
2. What does source passage retrieved mean?
3. Why is top-k held fixed across strategies?
4. What might happen when the query uses synonyms that do not occur in the passage?

**Next:** Notebook 05 runs an extractive QA model on retrieved passages.